In [7]:
import pandas as pd
import numpy as np
import requests

In [3]:
metadata_df = pd.read_csv(r"C:\ECG_Project\training\All excel files\ecg_metadata_inventory.csv")
print(metadata_df.shape)

# Official SNOMED -> diagnosis label mapping from the PhysioNet Challenge evaluation repo
mapping_url = "https://raw.githubusercontent.com/physionetchallenges/evaluation-2021/main/dx_mapping_scored.csv"
dx_mapping = pd.read_csv(mapping_url)
dx_mapping.head()

(87638, 10)


,Dx,SNOMEDCTCode,Abbreviation,CPSC,CPSC_Extra,StPetersburg,PTB,PTB_XL,Georgia,Chapman_Shaoxing,Ningbo,Total,Notes
0,atrial fibrillation,164889003,AF,1221,153,2,15,1514,570,1780,0,5255,NaN
1,atrial flutter,164890007,AFL,0,54,0,1,73,186,445,7615,8374,NaN
2,bundle branch block,6374002,BBB,0,0,1,20,0,116,0,385,522,NaN
3,bradycardia,426627000,Brady,0,271,11,0,0,6,0,7,295,NaN
4,complete left bundle branch block,733534002,CLBBB,0,0,0,0,0,0,0,213,213,We score 733534002 and 164909002 as the same d...


In [5]:
# See all available diagnosis classes with per-source breakdown
pd.set_option('display.max_rows', None)
dx_mapping[["Dx", "Abbreviation", "SNOMEDCTCode", "CPSC", "Georgia", "PTB_XL", 
            "Chapman_Shaoxing", "Ningbo", "Total"]]

,Dx,Abbreviation,SNOMEDCTCode,CPSC,Georgia,PTB_XL,Chapman_Shaoxing,Ningbo,Total
0,atrial fibrillation,AF,164889003,1221,570,1514,1780,0,5255
1,atrial flutter,AFL,164890007,0,186,73,445,7615,8374
2,bundle branch block,BBB,6374002,0,116,0,0,385,522
3,bradycardia,Brady,426627000,0,6,0,0,7,295
4,complete left bundle branch block,CLBBB,733534002,0,0,0,0,213,213
5,complete right bundle branch block,CRBBB,713427006,0,28,542,0,1096,1779
6,1st degree av block,IAVB,270492004,722,769,797,247,893,3534
7,incomplete right bundle branch block,IRBBB,713426002,0,407,1118,0,246,1857
8,left axis deviation,LAD,39732003,0,940,5146,382,1163,7631
9,left anterior fascicular block,LAnFB,445118002,0,180,1626,0,380,2186


In [1]:
def load_snomed_mapping():
    """Official SNOMED -> diagnosis label mapping from the PhysioNet Challenge evaluation repo."""
    mapping_url = "https://raw.githubusercontent.com/physionetchallenges/evaluation-2021/main/dx_mapping_scored.csv"
    dx_mapping = pd.read_csv(mapping_url)
    return dx_mapping

# Final shared taxonomy: SNOMED code -> class name
# Includes the CLBBB -> LBBB merge (733534002 scored same as 164909002)
CLASS_MAP = {
    "426783006": "NSR",     # Normal sinus rhythm
    "164889003": "AF",      # Atrial fibrillation
    "270492004": "IAVB",    # 1st degree AV block
    "39732003":  "LAD",     # Left axis deviation
    "164909002": "LBBB",    # Left bundle branch block
    "733534002": "LBBB",    # Complete LBBB -> merged into LBBB
    "698252002": "NSIVCB",  # Nonspecific intraventricular conduction disorder
    "284470004": "PAC",     # Premature atrial contraction
    "164917005": "QAb",     # Q wave abnormal
    "59118001":  "RBBB",    # Right bundle branch block
    "426177001": "SB",      # Sinus bradycardia
    "427084000": "STach",   # Sinus tachycardia
    "164934002": "TAb",     # T wave abnormal
}

FINAL_CLASSES = sorted(set(CLASS_MAP.values()))


def encode_labels(dx_codes_str: str) -> np.ndarray:
    """
    Takes the raw 'dx_codes' string (e.g. '426177001,55827005,164934002')
    and returns a multi-hot vector over FINAL_CLASSES.
    """
    label_vec = np.zeros(len(FINAL_CLASSES), dtype=np.int8)
    if pd.isna(dx_codes_str):
        return label_vec

    codes = [c.strip() for c in str(dx_codes_str).split(",")]
    for code in codes:
        if code in CLASS_MAP:
            class_name = CLASS_MAP[code]
            idx = FINAL_CLASSES.index(class_name)
            label_vec[idx] = 1
    return label_vec


def harmonize_labels(metadata_df: pd.DataFrame) -> pd.DataFrame:
    print(f"Final taxonomy ({len(FINAL_CLASSES)} classes):", FINAL_CLASSES)

    label_matrix = np.stack(metadata_df["dx_codes"].apply(encode_labels).values)
    label_df = pd.DataFrame(label_matrix, columns=FINAL_CLASSES)

    metadata_df = pd.concat([metadata_df.reset_index(drop=True), label_df], axis=1)

    metadata_df["has_label"] = metadata_df[FINAL_CLASSES].sum(axis=1) > 0
    print("Records with at least one relevant label:", metadata_df["has_label"].sum())
    print("Records dropped (no relevant label):", (~metadata_df["has_label"]).sum())

    metadata_df = metadata_df[metadata_df["has_label"]].reset_index(drop=True)

    coverage = metadata_df.groupby("source_hospital")[FINAL_CLASSES].sum()
    print(coverage)

    return metadata_df


if __name__ == "__main__":
    metadata_df = pd.read_csv(r"C:\ECG_Project\training\ecg_metadata_inventory.csv")
    print(metadata_df.shape)

    metadata_df = harmonize_labels(metadata_df)

    out_path = r"C:\ECG_Project\training\ecg_metadata_cleaned_labeled.csv"
    metadata_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

(87638, 10)
Final taxonomy (12 classes): ['AF', 'IAVB', 'LAD', 'LBBB', 'NSIVCB', 'NSR', 'PAC', 'QAb', 'RBBB', 'SB', 'STach', 'TAb']
Records with at least one relevant label: 72524
Records dropped (no relevant label): 15114
                    AF  IAVB   LAD  LBBB  NSIVCB    NSR   PAC  QAb  RBBB  \
source_hospital                                                            
chapman_shaoxing  1779   247   381   205     235   1822   258  234   454   
cpsc_2018         1221   722     0   236       0    918   616    0  1857   
cpsc_2018_extra    153   106     0    38       4      4    73    1     1   
georgia            570   769   940   231     203   1752   639  464   542   
ningbo               0   893  1163   248     536   6299  1054  828   195   
ptb-xl            1514   797  5146   536     789  18091   398  548     0   

                     SB  STach   TAb  
source_hospital                       
chapman_shaoxing   3882   1563  1870  
cpsc_2018             0      0     0  
cpsc_2018_ex